In [2]:
import pandas as pd
import numpy as np

import datetime
import os, sys
import importlib

import utils
importlib.reload(utils)

from utils import plot_series, plot_series_with_names, plot_series_bar
from utils import plot_dataframe
from utils import get_universe_adjusted_series, scale_weights_to_one, scale_to_book_long_short
from utils import generate_portfolio, backtest_portfolio
from utils import match_implementations

import plotly.graph_objects as go

In [3]:
data_dir = "/Users/yashzanwar/phase2_qrt_challenge/"

features = pd.read_parquet(os.path.join(data_dir, "features.parquet"))

universe = pd.read_parquet(os.path.join(data_dir, "universe_1m.parquet"))
 
returns = pd.read_parquet(os.path.join(data_dir, "returns.parquet"))

In [34]:
features_df = features.copy()

In [38]:
import ast
import pandas as pd

# Convert string → tuple
features_df.columns = features_df.columns.map(ast.literal_eval)

# Convert to MultiIndex
features_df.columns = pd.MultiIndex.from_tuples(features_df.columns)

# Optional: set names
features_df.columns.names = ["indicator", "ticker"]

In [39]:
features_df.index = universe.index

In [50]:
# Testing the backtest_portfolio function for a specific signal

def generate_portfolio_vectorized(
    entire_features: pd.DataFrame,
    universe: pd.DataFrame,
    start_date: str,
    end_date: str,
    signal_column: str
):
    # Validate date format
    try:
        start_dt = datetime.datetime.strptime(start_date, '%Y-%m-%d')
        end_dt = datetime.datetime.strptime(end_date, '%Y-%m-%d')
        cutoff_date = datetime.datetime.strptime('2005-01-01', '%Y-%m-%d')
    except ValueError:
        raise ValueError("start_date and end_date must be strings in 'YYYY-MM-DD' format.")

    # Ensure start_date is before end_date
    if start_dt >= end_dt:
        raise ValueError("start_date must be earlier than end_date.")

    # Ensure start_date is not before '2005-01-01'
    if start_dt < cutoff_date:
        raise ValueError("start_date must be later than '2005-01-01'.")

    # Get trading days within the date range
    trading_days = universe.index[(universe.index >= start_dt) & (universe.index <= end_dt)]

    if len(trading_days) == 0:
        raise ValueError("No Trading Days in the specified dates")
    
    portfolio = 0

    universe_boolean = universe.loc[:end_date].astype(bool)
    
    features_ = entire_features.loc[:end_date]

    signal1 = features_[signal_column].shift(1)
    signal1 = signal1.where(universe_boolean, np.nan)
    signal1 = signal1.rank(axis=1, method="min", ascending=True)
    signal1 = signal1.sub(signal1.mean(axis=1), axis=0)
    signal1 = signal1.div(signal1.abs().sum(axis=1), axis=0)
    signal1 = signal1.fillna(0)

    portfolio = -1 * signal1

    portfolio = portfolio.div(portfolio.abs().sum(axis=1), axis=0)
    
    return portfolio.fillna(0).loc[start_date:end_date]

In [51]:
benchmark_portfolio_vectorized = generate_portfolio_vectorized(
    features_df,
    universe,
    "2010-01-01",
    "2026-01-01",
    "ultimate_oscillator"
)

In [52]:
benchmark_portfolio_vectorized

ticker,AAPL,ABBV,ABT,ADI,AMAT,AMD,AMGN,AMZN,ANET,APH,...,VEEA,VNCE,VNRX,VNTG,WKHS,WYHG,YSXT,YXT,ZDAI,ZKIN
Date,,,,,,,,,,,,,,,,,,,,,
2010-01-04,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
2010-01-05,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
2010-01-06,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
2010-01-07,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
2010-01-08,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-24,-0.000448,-0.000533,-0.000072,0.000473,0.000311,0.000171,-0.000754,-0.000312,-0.000431,-0.000950,...,-0.0,-0.0,0.000166,-0.0,-0.0,-0.0,0.000623,-0.0,-0.0,-0.0
2025-12-26,-0.000737,-0.000195,0.000545,0.000690,0.000369,0.000057,-0.000623,-0.000671,-0.000233,-0.001004,...,-0.0,-0.0,-0.000030,-0.0,-0.0,-0.0,0.000901,-0.0,-0.0,-0.0
2025-12-29,-0.000388,-0.000406,0.000450,0.000699,0.000512,0.000303,-0.000550,-0.000679,-0.000213,-0.000954,...,-0.0,-0.0,-0.000177,-0.0,-0.0,-0.0,0.000695,-0.0,-0.0,-0.0


In [58]:
sr_vectorized, pnl_vectorized = backtest_portfolio(benchmark_portfolio_vectorized.loc["2010":"2019"], returns.loc["2010":"2019"], universe.loc["2010":"2020"], True, True)

ValueError: Shapes of portfolio, returns and universe are not algined

In [63]:
feature_results = pd.DataFrame(columns=["feature", "sharpe_ratio"])
feature_pnls = {}
for col in features_df.columns.get_level_values(0).unique():
    print(f"Testing feature: {col}")
    portfolio_vectorized = generate_portfolio_vectorized(
        features_df,
        universe,
        "2010-01-01",
        "2026-01-01",
        col
    )
    sr_vectorized, pnl_vectorized = backtest_portfolio(portfolio_vectorized.loc["2010":"2020"], returns.loc["2010":"2020"], universe.loc["2010":"2020"], False, False)
    new_row = pd.DataFrame([{
        "feature": col,
        "sharpe_ratio": sr_vectorized
    }])
    feature_pnls[col] = pnl_vectorized

    feature_results = pd.concat([feature_results, new_row], ignore_index=True)

Testing feature: relative_strength_index


/var/folders/wl/_t0cly451m96t1b5fcy2_0h80000gn/T/ipykernel_1249/3192833586.py:19: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



Testing feature: williams_r
Testing feature: rsi
Testing feature: volatility_20
Testing feature: volatility_60
Testing feature: trend_1_3
Testing feature: trend_5_20
Testing feature: trend_20_60
Testing feature: average_true_range
Testing feature: macd
Testing feature: macd_signal
Testing feature: macd_histogram
Testing feature: trix
Testing feature: commodity_channel_index
Testing feature: chande_momentum_oscillator
Testing feature: ichimoku_conversion
Testing feature: ichimoku_base
Testing feature: ichimoku_leading_a
Testing feature: ichimoku_leading_b
Testing feature: know_sure_thing
Testing feature: ultimate_oscillator
Testing feature: aroon_up
Testing feature: aroon_down
Testing feature: aroon_oscillator
Testing feature: stochastic_k
Testing feature: stochastic_d
Testing feature: on_balance_volume
Testing feature: ease_of_movement
Testing feature: chaikin_money_flow
Testing feature: accumulation_distribution_index
Testing feature: volume


In [80]:
top_signals = feature_results.sort_values(by="sharpe_ratio", ascending=False)
top_signals.head(5)

,feature,sharpe_ratio
15,ichimoku_conversion,0.893
17,ichimoku_leading_a,0.882
16,ichimoku_base,0.878
18,ichimoku_leading_b,0.862
8,average_true_range,0.818


In [81]:
selected_signals = pd.concat([feature_pnls['accumulation_distribution_index'], feature_pnls['stochastic_d'], feature_pnls['macd'], feature_pnls['ichimoku_leading_b'], feature_pnls['average_true_range']], axis=1)
selected_signals.columns = ['accumulation_distribution_index', 'stochastic_d', 'macd', 'ichimoku_leading_b', 'average_true_range']

In [82]:
selected_signals.corr()

,accumulation_distribution_index,stochastic_d,macd,ichimoku_leading_b,average_true_range
accumulation_distribution_index,1.000000,0.884629,0.873805,0.984761,0.951549
stochastic_d,0.884629,1.000000,0.914705,0.874599,0.855003
macd,0.873805,0.914705,1.000000,0.869308,0.850954
ichimoku_leading_b,0.984761,0.874599,0.869308,1.000000,0.971682
average_true_range,0.951549,0.855003,0.850954,0.971682,1.000000
